# 04 Train WLASL2000 Light V2 From Scratch

## What this notebook does

This notebook trains the Light V2 architecture from scratch. It gives us a baseline to compare against the fine-tuned WLASL1000 model.


## Model setup
```text
Input: MediaPipe keypoints + velocity
Input shape: (60, 516)
Model: BiGRU + Attention Light V2
Goal: main deployable WLASL2000 ASL recognition model
```

In [ ]:
from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

## 1. Set training configuration

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path("E:/Be_My_Ear")

DATASET_NAME = "WLASL2000"
PREFIX = "wlasl2000"

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
CLEAN_INDEX_FILE = BASE_DIR / f"{PREFIX}_clean_keypoint_index.csv"
LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / DATASET_NAME / f"asl_{PREFIX}_labels.json"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL" / DATASET_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR = PROJECT_ROOT / "reports" / f"phase1_{PREFIX}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dataset:", DATASET_NAME)
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Clean index exists:", CLEAN_INDEX_FILE.exists(), CLEAN_INDEX_FILE)
print("Label map exists:", LABEL_MAP_FILE.exists(), LABEL_MAP_FILE)
print("Model folder:", MODEL_DIR)
print("Report folder:", REPORT_DIR)

        MODEL_NAME = "light_v2_scratch"
        MODEL_DISPLAY_NAME = "WLASL2000 Light V2 From Scratch"
        TRAINING_TITLE = "Be My Ear - WLASL2000 Light V2 From Scratch"

        MODEL_PATH = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}.pt"
        HISTORY_PATH = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}_history.csv"
        NORM_STATS_PATH = MODEL_DIR / f"{PREFIX}_{MODEL_NAME}_train_norm_stats.npz"
        RESULT_FILE = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}_result_summary.csv"

        WLASL1000_MODEL_PATH = PROJECT_ROOT / "models" / "ASL" / "WLASL1000" / "bigru_attention_light_v2_wlasl1000.pt"

        BATCH_SIZE = 16
        EPOCHS = 90
        EARLY_STOPPING_PATIENCE = 18

        INPUT_SIZE = 516
        SEQUENCE_LENGTH = 60
        USE_VELOCITY = True

        HIDDEN_SIZE = 320
        NUM_LAYERS = 2
        DROPOUT = 0.35

        LEARNING_RATE = 2e-4

        print("Model path:", MODEL_PATH)
        print("Learning rate:", LEARNING_RATE)
        print("WLASL1000 checkpoint exists:", WLASL1000_MODEL_PATH.exists(), WLASL1000_MODEL_PATH)

## 2. Load clean data and create safe split

In [ ]:
df = pd.read_csv(CLEAN_INDEX_FILE)

NUM_CLASSES = df["label_id"].nunique()

print("Clean samples:", len(df))
print("Clean classes:", NUM_CLASSES)
print("Average samples per class:", round(len(df) / NUM_CLASSES, 2))

train_records = []
val_records = []
test_records = []
split_notes = []

for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(group)

    if n == 1:
        train_records.append(group)
        split_rule = "train_only"
    elif n == 2:
        train_records.append(group.iloc[:1])
        test_records.append(group.iloc[1:])
        split_rule = "1_train_1_test"
    elif n == 3:
        train_records.append(group.iloc[:1])
        val_records.append(group.iloc[1:2])
        test_records.append(group.iloc[2:])
        split_rule = "1_train_1_val_1_test"
    else:
        n_test = max(1, int(round(n * 0.15)))
        n_val = max(1, int(round(n * 0.15)))

        test_records.append(group.iloc[:n_test])
        val_records.append(group.iloc[n_test:n_test + n_val])
        train_records.append(group.iloc[n_test + n_val:])
        split_rule = "standard_70_15_15"

    split_notes.append({
        "label_id": int(label_id),
        "samples": int(n),
        "split_rule": split_rule
    })

train_df = pd.concat(train_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=SEED).reset_index(drop=True) if len(val_records) > 0 else pd.DataFrame(columns=df.columns)
test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True) if len(test_records) > 0 else pd.DataFrame(columns=df.columns)

split_notes_df = pd.DataFrame(split_notes)
split_notes_file = BASE_DIR / f"{PREFIX}_safe_split_notes.csv"
split_notes_df.to_csv(split_notes_file, index=False)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("Train classes:", train_df["label_id"].nunique())
print("Validation classes:", val_df["label_id"].nunique())
print("Test classes:", test_df["label_id"].nunique())

print("Saved split notes:", split_notes_file)
split_notes_df["split_rule"].value_counts()

## 3. Compute train-only normalisation

In [ ]:
def compute_train_normalisation_stats(train_dataframe):
    total_sum = None
    total_sq_sum = None
    total_count = 0

    for path in tqdm(train_dataframe["keypoint_path"], desc="Computing train mean/std"):
        arr = np.load(path).astype(np.float32)

        if total_sum is None:
            total_sum = arr.sum(axis=0)
            total_sq_sum = (arr ** 2).sum(axis=0)
        else:
            total_sum += arr.sum(axis=0)
            total_sq_sum += (arr ** 2).sum(axis=0)

        total_count += arr.shape[0]

    mean = total_sum / total_count
    variance = (total_sq_sum / total_count) - (mean ** 2)
    variance = np.maximum(variance, 1e-6)
    std = np.sqrt(variance)

    return mean.astype(np.float32), std.astype(np.float32)


train_mean, train_std = compute_train_normalisation_stats(train_df)
np.savez(NORM_STATS_PATH, mean=train_mean, std=train_std)

print("Saved normalisation stats:", NORM_STATS_PATH)
print("Mean shape:", train_mean.shape)
print("Std shape:", train_std.shape)

## 4. Build dataset class

In [ ]:
class WLASLKeypointDatasetLightAug(Dataset):
    def __init__(
        self,
        dataframe,
        mean,
        std,
        augment=False,
        noise_std=0.005,
        frame_mask_prob=0.03,
        temporal_shift_max=2
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mean = mean.reshape(1, -1).astype(np.float32)
        self.std = std.reshape(1, -1).astype(np.float32)
        self.augment = augment
        self.noise_std = noise_std
        self.frame_mask_prob = frame_mask_prob
        self.temporal_shift_max = temporal_shift_max

    def __len__(self):
        return len(self.dataframe)

    def _temporal_shift(self, keypoints):
        shift = np.random.randint(-self.temporal_shift_max, self.temporal_shift_max + 1)

        if shift == 0:
            return keypoints

        shifted = np.zeros_like(keypoints)

        if shift > 0:
            shifted[shift:] = keypoints[:-shift]
            shifted[:shift] = keypoints[0]
        else:
            shifted[:shift] = keypoints[-shift:]
            shifted[shift:] = keypoints[-1]

        return shifted

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        keypoints = np.load(row["keypoint_path"]).astype(np.float32)
        keypoints = (keypoints - self.mean) / (self.std + 1e-6)

        if self.augment:
            keypoints = self._temporal_shift(keypoints)

        velocity = np.zeros_like(keypoints, dtype=np.float32)
        velocity[1:] = keypoints[1:] - keypoints[:-1]

        features = np.concatenate([keypoints, velocity], axis=1).astype(np.float32)

        if self.augment:
            if self.noise_std > 0:
                features += np.random.normal(0, self.noise_std, features.shape).astype(np.float32)

            if self.frame_mask_prob > 0:
                frame_mask = np.random.rand(features.shape[0]) < self.frame_mask_prob
                features[frame_mask] = 0

        label = int(row["label_id"])

        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

## 5. Create balanced data loaders

In [ ]:
train_dataset = WLASLKeypointDatasetLightAug(
    train_df,
    train_mean,
    train_std,
    augment=True
)

val_dataset = WLASLKeypointDatasetLightAug(
    val_df,
    train_mean,
    train_std,
    augment=False
)

test_dataset = WLASLKeypointDatasetLightAug(
    test_df,
    train_mean,
    train_std,
    augment=False
)

class_counts = train_df["label_id"].value_counts().to_dict()
sample_weights = train_df["label_id"].map(lambda label: 1.0 / class_counts[label]).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))

print("Input batch shape:", x_batch.shape)
print("Label batch shape:", y_batch.shape)

## 6. Define Light V2 model

In [ ]:
class BiGRUAttentionLightV2(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.35):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        bi_hidden = hidden_size * 2

        self.attention = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)

        attention_scores = self.attention(gru_out).squeeze(-1)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)

        context = torch.sum(gru_out * attention_weights, dim=1)

        return self.classifier(context)


def build_light_v2_model(num_classes):
    return BiGRUAttentionLightV2(
        input_size=INPUT_SIZE,
        hidden_size=HIDDEN_SIZE,
        num_classes=num_classes,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    )

## 7. Initialise model

In [ ]:
model = build_light_v2_model(NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=5
)

def create_checkpoint_payload(epoch, best_val_f1, best_val_top5):
    return {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_f1": best_val_f1,
        "best_val_top5": best_val_top5,
        "num_classes": NUM_CLASSES,
        "input_size": INPUT_SIZE,
        "sequence_length": SEQUENCE_LENGTH,
        "use_velocity": USE_VELOCITY,
        "architecture": "BiGRUAttentionLightV2",
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "training_mode": "from_scratch"
    }

def build_model_for_eval(checkpoint):
    return build_light_v2_model(checkpoint.get("num_classes", NUM_CLASSES))

print("Parameters:", sum(p.numel() for p in model.parameters()))

## 8. Define training helper functions

In [ ]:
def top_k_accuracy(outputs, labels, k=5):
    _, top_k_preds = outputs.topk(k, dim=1)
    correct = top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds))
    return correct.any(dim=1).float().mean().item()


def run_epoch(model, loader, criterion, optimizer=None, phase="Train", epoch=1, total_epochs=1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0
    total_top1 = 0
    total_top3 = 0
    total_top5 = 0

    all_preds = []
    all_labels = []

    progress_bar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [{phase}]", leave=False)

    with torch.set_grad_enabled(is_train):
        for step, (x, y) in enumerate(progress_bar, start=1):
            x = x.to(device)
            y = y.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(x)
            loss = criterion(outputs, y)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds = torch.argmax(outputs, dim=1)

            batch_top1 = (preds == y).float().mean().item()
            batch_top3 = top_k_accuracy(outputs, y, 3)
            batch_top5 = top_k_accuracy(outputs, y, 5)

            total_loss += loss.item()
            total_top1 += batch_top1
            total_top3 += batch_top3
            total_top5 += batch_top5

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(y.detach().cpu().numpy())

            progress_bar.set_postfix({
                "step": f"{step}/{len(loader)}",
                "loss": f"{loss.item():.4f}",
                "top1": f"{batch_top1:.4f}",
                "top5": f"{batch_top5:.4f}"
            })

    avg_loss = total_loss / len(loader)
    avg_top1 = total_top1 / len(loader)
    avg_top3 = total_top3 / len(loader)
    avg_top5 = total_top5 / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, avg_top1, avg_top3, avg_top5, macro_f1


def top_k_accuracy_numpy(y_true, y_probs, k):
    correct = 0

    for true_label, prob in zip(y_true, y_probs):
        if true_label in np.argsort(prob)[-k:]:
            correct += 1

    return correct / len(y_true)


def collect_predictions(model, loader):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Collecting predictions"):
            x = x.to(device)
            outputs = model(x)

            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_labels.extend(y.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

## 9. Train model

In [ ]:
history = {k: [] for k in [
    "train_loss",
    "train_top1",
    "train_top3",
    "train_top5",
    "train_f1",
    "val_loss",
    "val_top1",
    "val_top3",
    "val_top5",
    "val_f1",
    "lr"
]}

best_val_f1 = 0.0
best_val_top5 = 0.0
epochs_without_improvement = 0

print("=" * 80)
print(TRAINING_TITLE)
print("=" * 80)
print("Device:", device)
print("Input shape:", (60, INPUT_SIZE))
print("Classes:", NUM_CLASSES)
print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Model path:", MODEL_PATH)
print("=" * 80)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 80)

    train_loss, train_top1, train_top3, train_top5, train_f1 = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
        phase="Training",
        epoch=epoch,
        total_epochs=EPOCHS
    )

    val_loss, val_top1, val_top3, val_top5, val_f1 = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None,
        phase="Validation",
        epoch=epoch,
        total_epochs=EPOCHS
    )

    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_top1"].append(train_top1)
    history["train_top3"].append(train_top3)
    history["train_top5"].append(train_top5)
    history["train_f1"].append(train_f1)

    history["val_loss"].append(val_loss)
    history["val_top1"].append(val_top1)
    history["val_top3"].append(val_top3)
    history["val_top5"].append(val_top5)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_top5 = val_top5
        epochs_without_improvement = 0

        torch.save(create_checkpoint_payload(epoch, best_val_f1, best_val_top5), MODEL_PATH)
        status = "Saved new best model"
    else:
        epochs_without_improvement += 1
        status = "No improvement"

    print(
        f"Train | Loss: {train_loss:.4f} | Top-1: {train_top1:.4f} | "
        f"Top-3: {train_top3:.4f} | Top-5: {train_top5:.4f} | F1: {train_f1:.4f}"
    )

    print(
        f"Val   | Loss: {val_loss:.4f} | Top-1: {val_top1:.4f} | "
        f"Top-3: {val_top3:.4f} | Top-5: {val_top5:.4f} | F1: {val_f1:.4f}"
    )

    print("Learning rate:", current_lr)
    print("Status:", status)
    print(f"Best Val F1: {best_val_f1:.4f}")
    print(f"Best Val Top-5: {best_val_top5:.4f}")
    print(f"Patience: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break

training_minutes = (time.time() - start_time) / 60

print("\nTraining completed.")
print("Training time minutes:", round(training_minutes, 2))
print("Best model saved:", MODEL_PATH)

## 10. Save history and evaluate test set

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print("Saved history:", HISTORY_PATH)
display(history_df.head())

checkpoint = torch.load(MODEL_PATH, map_location=device)

model = build_model_for_eval(checkpoint).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred, y_probs = collect_predictions(model, test_loader)

test_top1 = accuracy_score(y_true, y_pred)
test_top3 = top_k_accuracy_numpy(y_true, y_probs, 3)
test_top5 = top_k_accuracy_numpy(y_true, y_probs, 5)
test_macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("=" * 80)
print(MODEL_DISPLAY_NAME, "Test Evaluation")
print("=" * 80)
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")

result_df = pd.DataFrame([{
    "dataset": DATASET_NAME,
    "model": MODEL_DISPLAY_NAME,
    "clean_samples": len(df),
    "classes": NUM_CLASSES,
    "test_samples": len(test_df),
    "input_shape": f"(60, {INPUT_SIZE})",
    "features": "keypoints + velocity",
    "best_val_f1": checkpoint["best_val_f1"],
    "best_val_top5": checkpoint["best_val_top5"],
    "checkpoint_epoch": checkpoint["epoch"],
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_macro_f1,
    "model_path": str(MODEL_PATH),
    "history_path": str(HISTORY_PATH),
    "norm_stats_path": str(NORM_STATS_PATH)
}])

result_df.to_csv(RESULT_FILE, index=False)

report_result_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_result_summary.csv"
result_df.to_csv(report_result_file, index=False)

print("Saved result summary:")
print(RESULT_FILE)
print(report_result_file)

display(result_df)

## 11. Confidence threshold analysis

In [ ]:
if LABEL_MAP_FILE.exists():
    with open(LABEL_MAP_FILE, "r", encoding="utf-8") as f:
        label_map = json.load(f)

    id_to_gloss = {int(k): v["gloss"] for k, v in label_map.items()}
else:
    id_to_gloss = {}

test_df_reset = test_df.reset_index(drop=True)

prediction_records = []

for i in range(len(y_true)):
    true_id = int(y_true[i])
    pred_id = int(y_pred[i])
    confidence = float(y_probs[i][pred_id])
    top5_ids = np.argsort(y_probs[i])[-5:][::-1]

    prediction_records.append({
        "video_id": test_df_reset.iloc[i]["video_id"],
        "true_label_id": true_id,
        "true_gloss": id_to_gloss.get(true_id, str(true_id)),
        "predicted_label_id": pred_id,
        "predicted_gloss": id_to_gloss.get(pred_id, str(pred_id)),
        "confidence": confidence,
        "correct_top1": true_id == pred_id,
        "correct_top5": true_id in top5_ids,
        "top5_label_ids": ", ".join([str(int(x)) for x in top5_ids]),
        "top5_glosses": ", ".join([id_to_gloss.get(int(x), str(x)) for x in top5_ids]),
        "top5_probabilities": ", ".join([f"{float(y_probs[i][x]):.4f}" for x in top5_ids])
    })

predictions_df = pd.DataFrame(prediction_records)

predictions_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_test_predictions.csv"
predictions_df.to_csv(predictions_file, index=False)

threshold_records = []

for threshold_value in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    confident = predictions_df[predictions_df["confidence"] >= threshold_value]

    threshold_records.append({
        "confidence_threshold": threshold_value,
        "coverage": len(confident) / len(predictions_df),
        "top1_accuracy_on_confident_samples": confident["correct_top1"].mean() if len(confident) else np.nan,
        "top5_accuracy_on_confident_samples": confident["correct_top5"].mean() if len(confident) else np.nan,
        "num_confident_samples": len(confident)
    })

threshold_df = pd.DataFrame(threshold_records)

threshold_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_confidence_threshold_analysis.csv"
threshold_df.to_csv(threshold_file, index=False)

print("Saved predictions:", predictions_file)
print("Saved confidence threshold analysis:", threshold_file)

display(threshold_df)

## What to send me after this notebook
Send me the final summary table with:

```text
Best Val F1
Best Val Top-5
Test Top-1
Test Top-3
Test Top-5
Test Macro F1
Confidence threshold table
```